In [ ]:
%matplotlib inline
import pathlib as pl
import numpy as np
import sys
import xugrid
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import flopy
from flopy.export.shapefile_utils import recarray2shp
from flopy.utils.geometry import Polygon
import flopy.plot.styles as styles

# Load MF Simulation

In [ ]:
to_crs_projection = "EPSG:32618"
ws = pl.Path("../pj_2018_adjust_FINAL/base/")
sim = flopy.mf6.MFSimulation.load(sim_ws=ws, load_only= ['dis''tdis','chd_coast',"ghb"],use_pandas=False)
sim.model_names
gwf = sim.get_model("gwf")
gwf.modelgrid.crs = "EPSG:4456"

# Create the shapefile of the chd surface data

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_chd_4456.shp")
chds = gwf.get_package("chd_coast")
chd_spd = pd.DataFrame(chds.stress_period_data.get_data(0))
print(f'{chd_spd.shape[0]} CHD cells to be coupled')

In [ ]:
vertices = []
for k, i, j in chd_spd["cellid"]:
    vertices.append(gwf.modelgrid.get_cell_vertices(i, j))
# create polygons for the shapefile
polygons = [Polygon(vrt) for vrt in vertices]
len(polygons)

In [ ]:
layer = [k for k, i, j in chd_spd["cellid"]]
row = [i for k, i, j in chd_spd["cellid"]]
column = [j for k, i, j in chd_spd["cellid"]]

chd_spd["layer"] = layer
chd_spd["row"] = row
chd_spd["column"] = column
chd_spd.drop(columns=["cellid"], inplace=True)
chd_spd["chd_no"] = chd_spd.index
chd_spd

In [ ]:
# Using flopy.utils.recarray2shp() to write a shapefile (.shp) and save it to the 'fpth'
# BNB note -  I don't think the line below actually assigns the crs
recarray2shp(chd_spd.to_records(index=False), geoms=polygons, shpname=fpth, crs=gwf.modelgrid.crs)

In [ ]:
# Read in the shapefile that was saved in the previous cell
gdf = gpd.read_file(fpth)
print(gdf.crs)
gdf.crs = gwf.modelgrid.crs
print(gdf.crs)
fig,ax=plt.subplots()
gdf.plot(ax=ax,column='boundname')

## Remove most perimeter CHDs for coupling

In [ ]:
# gdf = gdf.copy() 
# # create buffer around CHDs in the bay
# chd_buffer = gdf[gdf['boundname'] == 'bay'].buffer(1).unary_union
# # locate perimeter chd's that intersect the chd bay buffer, and remane them to 'perimeter coastal' to differentiate them from the other perimeter CHDs
# gdf.loc[(gdf['boundname'] == 'perimeter-heads') & (gdf.geometry.intersects(chd_buffer)),'boundname'] = 'perimeter coastal'
# # filter the perimeter CHDs out of the gdf
# gdf = gdf[gdf['boundname'] != 'perimeter-heads']
# # remove any cells where layer is greate than 0, because we only want coupling in the first layer. It should only be filtering out perimeter coastal CHDs because the other CHD and GHBs are only assigned at the first layer.
# gdf = gdf[gdf['layer']==0]
# gdf=gdf.reset_index()
# gdf["chd_no"] = gdf.index
# gdf.crs
# gdf.plot(column='boundname')

In [ ]:
# chd_spd = gdf.copy()
# chd_spd = chd_spd.drop(columns = ['geometry'])
# chd_spd.to_csv('../gis/PJ/pj_SurfaceChd.csv')

In [ ]:
# chd_spd['boundname'].unique()

In [ ]:
# gdf.to_file(fpth)

## Reproject the chd surface data shapefile to UTM zone 18N

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_CHD_utm18n.shp")

In [ ]:
# BNB Note: line commented below does not work to assign crs
#gdf.to_crs(to_crs_projection).to_file(fpth)

gdf = gdf.to_crs(to_crs_projection)
print(gdf.crs)
gdf.to_file(fpth)

assert gdf.crs == to_crs_projection

# Create the shapefile of the ghb data

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_ghb_4456.shp")
ghb = gwf.get_package("ghb")
ghb_spd = pd.DataFrame(ghb.stress_period_data.get_data(0))
ghb_spd["boundname"] = "estuary"
ghb_spd.rename(columns={"bhead": "head"}, errors="raise", inplace=True)
ghb_spd

In [ ]:
vertices = []
for k, i, j in ghb_spd["cellid"]:
    vertices.append(gwf.modelgrid.get_cell_vertices(i, j))
# create polygons for the shapefile
polygons = [Polygon(vrt) for vrt in vertices]
len(polygons)

In [ ]:
layer = [k for k, i, j in ghb_spd["cellid"]]
row = [i for k, i, j in ghb_spd["cellid"]]
column = [j for k, i, j in ghb_spd["cellid"]]

ghb_spd["layer"] = layer
ghb_spd["row"] = row
ghb_spd["column"] = column
ghb_spd.drop(columns=["cellid"], inplace=True)
ghb_spd["ghb_no"] = ghb_spd.index

In [ ]:
recarray2shp(ghb_spd.to_records(index=False), geoms=polygons, shpname=fpth, crs=gwf.modelgrid.crs)

In [ ]:
gdf = gpd.read_file(fpth)
print(gdf.crs)
gdf.crs = gwf.modelgrid.crs
print(gdf.crs)
gdf.plot()

## Reproject the ghb data shapefile to UTM zone 18N

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_ghb_utm18n.shp")

In [ ]:
# BNB Note: line commented below does not work to assign crs
#gdf.to_crs(to_crs_projection).to_file(fpth)

gdf = gdf.to_crs(to_crs_projection)
print(gdf.crs)
gdf.to_file(fpth)

assert gdf.crs == to_crs_projection


# Concatenate the two dataframes to create a shapefile for plotting

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_chd_ghb_plot_4456.shp")

In [ ]:
spd = pd.concat([chd_spd, ghb_spd])

In [ ]:
spd = spd.assign(bnd_no=spd.chd_no.mask(spd.chd_no.isnull(), spd.ghb_no))

In [ ]:
spd["bnd_no"] = spd["bnd_no"].astype(int)

In [ ]:
vertices = []
for i, j in zip(spd["row"], spd["column"]):
    vertices.append(gwf.modelgrid.get_cell_vertices(i, j))
polygons = [Polygon(vrt) for vrt in vertices]
len(polygons)

In [ ]:
recarray2shp(spd.to_records(index=False), geoms=polygons, shpname=fpth, crs=gwf.modelgrid.crs)

In [ ]:
gdf = gpd.read_file(fpth)
print(gdf.crs)
gdf.crs = gwf.modelgrid.crs
print(gdf.crs)

In [ ]:
gdf.plot(column="boundname")

## Reproject the chd and ghb data shapefile to UTM zone 18N

In [ ]:
fpth = pl.Path("../gis/PJ/PJ_coast_utm18n.shp")

In [ ]:
# BNB Note: line commented below does not work to assign crs
#gdf.to_crs(to_crs_projection).to_file(fpth)

gdf = gdf.to_crs(to_crs_projection)
print(gdf.crs)
gdf.to_file(fpth)

assert gdf.crs == to_crs_projection